# D3 + D4: Train & Eval Main Model (SigLIP-2 + BGE-M3 + FusionEncoder)

Notebook này gộp D3 (training) và D4 (evaluation) trong cùng 1 file.

Nguyên tắc:
- Import và dùng lại code từ file khác: source/dataset.py, source/model.py.
- Không định nghĩa class mới trong notebook.
- Chỉ cung cấp code, bạn tự chạy từng cell khi cần.

In [1]:
import gc
import json
import sys
from datetime import datetime
from pathlib import Path
import os

import numpy as np
import torch
import torch.nn.functional as F
from tqdm.auto import tqdm
from transformers import AutoModel, AutoTokenizer, get_cosine_schedule_with_warmup

cwd = Path.cwd().resolve()
if (cwd / 'source').exists():
    BASE_DIR = cwd
elif (cwd.parent / 'source').exists():
    BASE_DIR = cwd.parent
else:
    raise RuntimeError(f'Cannot locate project root from cwd={cwd}')

SOURCE_DIR = BASE_DIR / 'source'
if str(SOURCE_DIR) not in sys.path:
    sys.path.insert(0, str(SOURCE_DIR))

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('BASE_DIR :', BASE_DIR)
print('DEVICE   :', DEVICE)
if torch.cuda.is_available():
    print('GPU      :', torch.cuda.get_device_name(0))

/home/urlab/miniconda3/envs/uav_ai/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


BASE_DIR : /media/urlab/KINGSTON/aic
DEVICE   : cuda
GPU      : NVIDIA GeForce RTX 5060 Ti


In [2]:
from dataset import get_dataloader
from model import FusionEncoder

print('Imported get_dataloader and FusionEncoder successfully')

Imported get_dataloader and FusionEncoder successfully


In [3]:
CFG = {
    'base_dir': BASE_DIR,
    'output_dir': BASE_DIR / 'source' / 'fusion_output',

    # data config (D1)
    'max_frames': 25,
    'max_segments': 15,
    'max_seg_tokens': 128,

    # model config (D2)
    'dim': 1024,
    'vis_dim': 1152,
    'n_layers': 2,
    'n_heads': 16,
    'n_kv_heads': 4,

    # train config (D3)
    'epochs': 20,
    'batch_size': 4,
    'num_workers': os.cpu_count(),
    'lr': 1e-4,
    'weight_decay': 0.01,
    'warmup_ratio': 0.1,
    'grad_clip': 1.0,
    'lambda_narr': 0.5,
    'beta_hard': 0.3,
    'gamma_mrl': 0.1,
    'train_query_max_length': 512,

    # eval config (D4)
    'dual_softmax_tau': 0.01,
    'eval_query_max_length': 256,
    'eval_query_batch_size': 16,
    'baseline_result_file': BASE_DIR / 'source' / 'narvid_baseline_results.json',
}

CFG['output_dir'].mkdir(parents=True, exist_ok=True)
print('Output dir:', CFG['output_dir'])

Output dir: /media/urlab/KINGSTON/aic/source/fusion_output


In [4]:
# Load BGE-M3 local weights for online query encoding (frozen)
bgem3_path = BASE_DIR / 'features' / 'weights' / 'bgem3'
bgem3_tokenizer = AutoTokenizer.from_pretrained(str(bgem3_path), local_files_only=True)
bgem3_model = AutoModel.from_pretrained(str(bgem3_path), local_files_only=True)
bgem3_model.eval().to(DEVICE)
for p in bgem3_model.parameters():
    p.requires_grad = False

print('Loaded BGE-M3 from:', bgem3_path)

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 3126.92it/s, Materializing param=pooler.dense.weight]                               


Loaded BGE-M3 from: /media/urlab/KINGSTON/aic/features/weights/bgem3


In [5]:
train_loader = get_dataloader(
    split='train',
    base_dir=CFG['base_dir'],
    batch_size=CFG['batch_size'],
    num_workers=CFG['num_workers'],
    max_frames=CFG['max_frames'],
    max_segments=CFG['max_segments'],
    max_seg_tokens=CFG['max_seg_tokens'],
)

val_loader = get_dataloader(
    split='val',
    base_dir=CFG['base_dir'],
    batch_size=CFG['batch_size'],
    num_workers=CFG['num_workers'],
    max_frames=CFG['max_frames'],
    max_segments=CFG['max_segments'],
    max_seg_tokens=CFG['max_seg_tokens'],
)

test_loader = get_dataloader(
    split='test',
    base_dir=CFG['base_dir'],
    batch_size=CFG['batch_size'],
    num_workers=CFG['num_workers'],
    max_frames=CFG['max_frames'],
    max_segments=CFG['max_segments'],
    max_seg_tokens=CFG['max_seg_tokens'],
)

print('len(train_loader.dataset)=', len(train_loader.dataset))
print('len(val_loader.dataset)=', len(val_loader.dataset))
print('len(test_loader.dataset)=', len(test_loader.dataset))

len(train_loader.dataset)= 9222
len(val_loader.dataset)= 9222
len(test_loader.dataset)= 9222


In [6]:
model = FusionEncoder(
    dim=CFG['dim'],
    vis_dim=CFG['vis_dim'],
    n_layers=CFG['n_layers'],
    n_heads=CFG['n_heads'],
    n_kv_heads=CFG['n_kv_heads'],
).to(DEVICE)

optimizer = torch.optim.AdamW(model.parameters(), lr=CFG['lr'], weight_decay=CFG['weight_decay'])
total_steps = max(1, len(train_loader) * CFG['epochs'])
warmup_steps = int(total_steps * CFG['warmup_ratio'])
scheduler = get_cosine_schedule_with_warmup(optimizer, warmup_steps, total_steps)
scaler = torch.amp.GradScaler('cuda' if DEVICE.type == 'cuda' else 'cpu')

print('Model params:', sum(p.numel() for p in model.parameters()) / 1e6, 'M')
print('Total steps :', total_steps)

Model params: 45.224962 M
Total steps : 46120


In [7]:
def encode_queries(
    texts,
    tokenizer,
    encoder_model,
    device,
    max_length=512,
    batch_size=None,
    show_progress=False,
    progress_desc='Encode queries',
):
    if len(texts) == 0:
        hidden_dim = int(getattr(encoder_model.config, 'hidden_size', 1024))
        return torch.empty((0, hidden_dim), device=device)

    if batch_size is None or batch_size <= 0:
        batch_size = len(texts)

    hidden_dim = int(getattr(encoder_model.config, 'hidden_size', 1024))
    ranges = range(0, len(texts), batch_size)
    if show_progress:
        total_batches = (len(texts) + batch_size - 1) // batch_size
        ranges = tqdm(ranges, total=total_batches, desc=progress_desc, leave=False, dynamic_ncols=True)

    all_embs = []
    with torch.inference_mode():
        for start in ranges:
            chunk = texts[start:start + batch_size]
            enc = tokenizer(
                chunk,
                padding=True,
                truncation=True,
                max_length=max_length,
                return_tensors='pt',
            ).to(device)

            if device.type == 'cuda':
                with torch.amp.autocast(device_type='cuda', enabled=True, dtype=torch.bfloat16):
                    out = encoder_model(**enc, return_dict=True)
                    h = out.last_hidden_state
            else:
                out = encoder_model(**enc, return_dict=True)
                h = out.last_hidden_state

            m = enc['attention_mask'].unsqueeze(-1).to(h.dtype)
            pooled = (h * m).sum(dim=1) / m.sum(dim=1).clamp_min(1e-6)
            all_embs.append(F.normalize(pooled.float(), p=2, dim=-1))

    if len(all_embs) == 0:
        return torch.empty((0, hidden_dim), device=device)
    return torch.cat(all_embs, dim=0)

def symmetric_infonce(q, d, temperature_param):
    tau = torch.exp(temperature_param).clamp_min(1e-6)
    sim = torch.matmul(q, d.T) / tau
    labels = torch.arange(sim.size(0), device=sim.device)
    loss_q2d = F.cross_entropy(sim, labels)
    loss_d2q = F.cross_entropy(sim.T, labels)
    return 0.5 * (loss_q2d + loss_d2q)

def retrieval_loss(q_hat, e_plus, e_narr, temperature_param, hard_neg_embs=None):
    l_base = symmetric_infonce(q_hat, e_plus, temperature_param)
    l_narr = symmetric_infonce(q_hat, e_narr, temperature_param)

    l_hard = torch.tensor(0.0, device=q_hat.device)
    if hard_neg_embs is not None:
        # hard_neg_embs: [B, K, D]
        q = q_hat.unsqueeze(1)  # [B,1,D]
        neg_scores = torch.sum(q * hard_neg_embs, dim=-1)  # [B,K]
        pos_scores = torch.sum(q_hat * e_plus, dim=-1, keepdim=True)  # [B,1]
        margin = 0.1
        l_hard = torch.relu(neg_scores - pos_scores + margin).mean()

    # optional MRL over truncated dims
    l_mrl = torch.tensor(0.0, device=q_hat.device)
    for d in [64, 128, 256, 512, 1024]:
        l_mrl = l_mrl + symmetric_infonce(q_hat[:, :d], e_plus[:, :d], temperature_param)
    l_mrl = l_mrl / 5.0

    return l_base + CFG['lambda_narr'] * l_narr + CFG['beta_hard'] * l_hard + CFG['gamma_mrl'] * l_mrl

In [8]:
def _encode_hard_negs(hard_neg_texts_batch):
    # hard_neg_texts_batch: List[List[str]], size B, each up to 4 texts
    b = len(hard_neg_texts_batch)
    k = max((len(x) for x in hard_neg_texts_batch), default=0)
    if k == 0:
        return None

    flat = []
    for negs in hard_neg_texts_batch:
        pad = negs + [''] * (k - len(negs))
        flat.extend(pad)

    emb = encode_queries(
        flat,
        bgem3_tokenizer,
        bgem3_model,
        DEVICE,
        max_length=CFG['train_query_max_length'],
    )
    emb = emb.view(b, k, -1)
    return emb

def train_one_epoch(epoch_idx):
    model.train()
    running_loss = 0.0
    total_steps = max(1, len(train_loader))

    pbar = tqdm(
        train_loader,
        total=total_steps,
        desc=f'Train Epoch {epoch_idx:02d}',
        leave=False,
        dynamic_ncols=True,
    )

    for step, batch in enumerate(pbar, start=1):
        q_hat = encode_queries(
            batch['query_text'],
            bgem3_tokenizer,
            bgem3_model,
            DEVICE,
            max_length=CFG['train_query_max_length'],
        )
        hard_neg_embs = _encode_hard_negs(batch['hard_neg_texts'])

        visual_features = batch['visual_features'].to(DEVICE)
        visual_mask = batch['visual_mask'].to(DEVICE)
        frame_timestamps = batch['frame_timestamps'].to(DEVICE)

        token_reprs = batch['token_reprs'].to(DEVICE)
        token_mask = batch['token_mask'].to(DEVICE)
        segment_pooled = batch['segment_pooled'].to(DEVICE)
        segment_mask = batch['segment_mask'].to(DEVICE)
        seg_timestamps = batch['seg_timestamps'].to(DEVICE)

        with torch.amp.autocast(
            device_type='cuda' if DEVICE.type == 'cuda' else 'cpu',
            enabled=(DEVICE.type == 'cuda'),
            dtype=torch.bfloat16 if DEVICE.type == 'cuda' else torch.float32,
        ):
            e_plus, e_narr, temperature = model(
                seg_tokens=token_reprs,
                seg_pooled=segment_pooled,
                visual_features=visual_features,
                seg_timestamps=seg_timestamps,
                frame_timestamps=frame_timestamps,
                seg_mask=segment_mask,
                visual_mask=visual_mask,
                query_emb=q_hat,
                token_mask=token_mask,
            )
            loss = retrieval_loss(q_hat, e_plus, e_narr, temperature, hard_neg_embs=hard_neg_embs)

        optimizer.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), CFG['grad_clip'])
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

        running_loss += float(loss.item())
        avg_loss = running_loss / step
        pbar.set_postfix(loss=f'{avg_loss:.4f}')

    return running_loss / total_steps

In [9]:
def precompute_all_documents(loader, split='val'):
    model.eval()
    doc_embs = []
    shot_ids = []
    total_batches = max(1, len(loader))

    pbar = tqdm(
        loader,
        total=total_batches,
        desc=f'[{split}] Encode docs',
        leave=False,
        dynamic_ncols=True,
    )

    with torch.inference_mode():
        for batch in pbar:
            visual_features = batch['visual_features'].to(DEVICE)
            visual_mask = batch['visual_mask'].to(DEVICE)
            frame_timestamps = batch['frame_timestamps'].to(DEVICE)

            token_reprs = batch['token_reprs'].to(DEVICE)
            token_mask = batch['token_mask'].to(DEVICE)
            segment_pooled = batch['segment_pooled'].to(DEVICE)
            segment_mask = batch['segment_mask'].to(DEVICE)
            seg_timestamps = batch['seg_timestamps'].to(DEVICE)

            with torch.amp.autocast(
                device_type='cuda' if DEVICE.type == 'cuda' else 'cpu',
                enabled=(DEVICE.type == 'cuda'),
                dtype=torch.bfloat16 if DEVICE.type == 'cuda' else torch.float32,
            ):
                e_plus, _, _ = model(
                    seg_tokens=token_reprs,
                    seg_pooled=segment_pooled,
                    visual_features=visual_features,
                    seg_timestamps=seg_timestamps,
                    frame_timestamps=frame_timestamps,
                    seg_mask=segment_mask,
                    visual_mask=visual_mask,
                    query_emb=None,
                    token_mask=token_mask,
                )

            doc_embs.append(e_plus.detach().cpu())
            shot_ids.extend(batch['shot_id'])
            pbar.set_postfix(docs=len(shot_ids))

    if len(doc_embs) == 0:
        return torch.empty((0, CFG['dim'])), []

    doc_embs = torch.cat(doc_embs, dim=0)
    return F.normalize(doc_embs, p=2, dim=-1), shot_ids

def load_queries_for_eval(split='test'):
    q_items = []
    data_dir = BASE_DIR / 'data' / split
    for jf in sorted(data_dir.glob('*.json')):
        video_id = jf.stem
        rows = json.loads(jf.read_text(encoding='utf-8'))
        grouped = {}
        for r in rows:
            sid = str(r['id']).zfill(3)
            if sid not in grouped:
                grouped[sid] = str(r['positive']).strip()
        for sid, q in grouped.items():
            q_items.append({'shot_id': f'{video_id}_{sid}', 'query': q})
    return q_items

def build_sim_matrix(query_embs, doc_embs):
    return torch.matmul(query_embs, doc_embs.T)

def dual_softmax(sim_matrix, tau=0.01):
    s = sim_matrix / tau
    s_row = torch.softmax(s, dim=1)
    s_col = torch.softmax(s, dim=0)
    return s_row * s_col

def compute_all_metrics(sim_matrix, gt_indices):
    sim_np = sim_matrix.detach().cpu().numpy()
    ranks = []
    for i, gt in enumerate(gt_indices):
        sorted_idx = np.argsort(-sim_np[i])
        rank = int(np.where(sorted_idx == gt)[0][0]) + 1
        ranks.append(rank)

    ranks = np.asarray(ranks)
    return {
        'R1': 100.0 * float(np.mean(ranks <= 1)),
        'R5': 100.0 * float(np.mean(ranks <= 5)),
        'R10': 100.0 * float(np.mean(ranks <= 10)),
        'MdR': float(np.median(ranks)),
        'MnR': float(np.mean(ranks)),
        'SumR': 100.0 * float(np.mean(ranks <= 1) + np.mean(ranks <= 5) + np.mean(ranks <= 10)),
    }

def evaluate(split='val'):
    loader = val_loader if split == 'val' else test_loader
    doc_embs, shot_ids = precompute_all_documents(loader, split=split)

    q_items = load_queries_for_eval(split)
    q_texts = [x['query'] for x in q_items]
    q_embs = encode_queries(
        q_texts,
        bgem3_tokenizer,
        bgem3_model,
        DEVICE,
        max_length=CFG['eval_query_max_length'],
        batch_size=CFG['eval_query_batch_size'],
        show_progress=True,
        progress_desc=f'[{split}] Encode queries',
    )

    sim = build_sim_matrix(q_embs, doc_embs.to(DEVICE))
    sim_dsl = dual_softmax(sim, tau=CFG['dual_softmax_tau'])

    shot_to_idx = {sid: i for i, sid in enumerate(shot_ids)}
    filtered_q = [x for x in q_items if x['shot_id'] in shot_to_idx]
    if len(filtered_q) == 0:
        raise RuntimeError('No GT shot ids overlap with doc embeddings. Check split/data alignment.')

    keep_idx = [i for i, x in enumerate(q_items) if x['shot_id'] in shot_to_idx]
    sim_kept = sim_dsl[keep_idx]
    gt_indices = [shot_to_idx[x['shot_id']] for x in filtered_q]

    return compute_all_metrics(sim_kept, gt_indices)

In [10]:
best_r1 = -1.0
history = []
best_ckpt = CFG['output_dir'] / 'best_fusion_model.pth'

for epoch in range(1, CFG['epochs'] + 1):
    train_loss = train_one_epoch(epoch)

    if DEVICE.type == 'cuda':
        gc.collect()
        torch.cuda.empty_cache()

    val_metrics = evaluate(split='val')

    history.append({
        'epoch': epoch,
        'train_loss': train_loss,
        **val_metrics,
    })

    if val_metrics['R1'] > best_r1:
        best_r1 = val_metrics['R1']
        torch.save(model.state_dict(), best_ckpt)

    print(f"Epoch {epoch:02d}/{CFG['epochs']} | loss={train_loss:.4f} | val R@1={val_metrics['R1']:.2f} | val R@5={val_metrics['R5']:.2f} | val R@10={val_metrics['R10']:.2f}")

(CFG['output_dir'] / 'training_log.json').write_text(json.dumps(history, indent=2), encoding='utf-8')
print('Best Val R@1:', best_r1)
print('Saved best model:', best_ckpt)
print('Saved log:', CFG['output_dir'] / 'training_log.json')

Epoch 01/20 | loss=2.2180 | val R@1=30.17 | val R@5=51.41 | val R@10=60.53


Epoch 02/20 | loss=2.2091 | val R@1=36.62 | val R@5=58.75 | val R@10=67.20


Epoch 03/20 | loss=2.2022 | val R@1=43.61 | val R@5=66.96 | val R@10=75.02


Epoch 04/20 | loss=2.1927 | val R@1=47.73 | val R@5=70.84 | val R@10=77.49


Epoch 05/20 | loss=2.1805 | val R@1=46.70 | val R@5=68.95 | val R@10=76.28


...

Epoch 16/20 | loss=1.9317 | val R@1=58.86 | val R@5=79.94 | val R@10=84.92


Epoch 17/20 | loss=1.9209 | val R@1=58.83 | val R@5=79.77 | val R@10=84.72


Epoch 18/20 | loss=1.9158 | val R@1=59.27 | val R@5=80.16 | val R@10=85.13


Epoch 19/20 | loss=1.9113 | val R@1=59.07 | val R@5=79.98 | val R@10=84.97


Epoch 20/20 | loss=1.9113 | val R@1=59.09 | val R@5=80.04 | val R@10=85.00
Best Val R@1: 59.27130774235524
Saved best model: /media/urlab/KINGSTON/aic/source/fusion_output/best_fusion_model.pth
Saved log: /media/urlab/KINGSTON/aic/source/fusion_output/training_log.json


In [11]:
if best_ckpt.exists():
    state = torch.load(best_ckpt, map_location=DEVICE)
    model.load_state_dict(state)

test_metrics = evaluate(split='test')

if CFG['baseline_result_file'].exists():
    baseline = json.loads(CFG['baseline_result_file'].read_text(encoding='utf-8'))
else:
    baseline = {'R1': 2.74, 'R5': 7.53, 'R10': 12.33, 'SumR': 22.60}

print('=' * 65)
print('MODEL CHINH vs NARVID BASELINE')
print(f"NarVid: R@1={baseline['R1']:.2f}  R@5={baseline['R5']:.2f}  R@10={baseline['R10']:.2f}  SumR={baseline['SumR']:.2f}")
print(f"Custom: R@1={test_metrics['R1']:.2f}  R@5={test_metrics['R5']:.2f}  R@10={test_metrics['R10']:.2f}  SumR={test_metrics['SumR']:.2f}")

result = {
    'timestamp': datetime.utcnow().isoformat() + 'Z',
    'model': 'SigLIP2+BGE-M3+FusionEncoder',
    **test_metrics,
}
out_json = BASE_DIR / 'source' / 'custom_model_results.json'
out_json.write_text(json.dumps(result, indent=2), encoding='utf-8')
print('Saved:', out_json)

MODEL CHINH vs NARVID BASELINE
NarVid: R@1=2.74  R@5=7.53  R@10=12.33  SumR=22.60
Custom: R@1=63.24  R@5=83.24  R@10=87.77  SumR=234.24
Saved: /media/urlab/KINGSTON/aic/source/custom_model_results.json


## Thu tu chay goi y

1. Chay Cell 1 -> Cell 8 de setup environment, data, model.
2. Chay Cell 9 de train va val theo epoch.
3. Chay Cell 10 de test va so sanh baseline.

Notebook nay dung lai class/ham tu source/dataset.py va source/model.py.